In [1]:
import pandas as pd

In [6]:
data=pd.read_excel(r"R:\adarsha\power_bi_legit_columns\Files-solution (1)\Files\ABC Store.xlsx")
data

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22
0,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...
1,SNO.,BILL Date,BILL NO.,AGENT NAME,COMPANY NAME,ITEM CODE,ITEM NAME,SHADE NAME,PACK / SIZE,SALE QUANTITY,...,CATEGORY.,SEASON.,CODE.,SUB CATEGORY.,GROSS AMOUNT,CD(%),CD VALUE,NET AMOUNT,LOT NUMBER,M.R.P.
2,1,01/12/2022,ASV-24598,NIL,LEVI`S,633731222368,8XR511-508 508,508,7X,1,...,BOYS WEAR,AW22,WLL,JEANS,2095,13.75,-288.0625,1806.9375,RSTIDL/2211/1026,2095
3,2,01/12/2022,ASV-24598,NIL,NIKE,825663750167,86J843-U90 `U90,`U90,7,1,...,BOYS WEAR,AW22,WLL,SWEATSHIRT,2995,13.75,-411.8125,2583.1875,RSTIDL/2209/0603,2995
4,3,01/12/2022,ASV-24600,NIL,NIKE,742728718507,76H992-U1A `U1A,`U1A,2T,-1,...,BOYS WEAR,AW22,WLL,JOGGERS,-1695,NaN,0,-1695,RSTIDL/2209/0603,1695
5,4,01/12/2022,ASV-24600,NIL,NIKE,742728721637,76H926 U9J,U9J,3T,1,...,BOYS WEAR,AW21,WLI,JOGGER,1695,NaN,0,1695,RSTIDL/2209/0603,1695
6,5,01/12/2022,ASV-24604,NIL,JORDAN,807421814823,95B239-023 23,23,L,1,...,BOYS WEAR,AW22,WLL,T-SHIRT,1695,12.2,-206.79,1488.21,RSTIDL/2207/2598,1695
7,6,01/12/2022,ASV-24604,NIL,JORDAN,825664083455,95B958-023 23,23,L,1,...,BOYS WEAR,AW22,WLL,JACKET,6495,12.2,-792.39,5702.61,RSTIDL/2211/1026,6495
8,7,02/12/2022,ASV-24778,NIL,NIKE,677838669718,36G461-W6U W6U,W6U,4,1,...,GIRLS WEAR,AW22,WLL,JACKET,5995,11.67,-699.6165,5295.3835,RSTIDL/2211/1026,5995
9,8,03/12/2022,ASV-24853,NIL,LEVI`S,825663570468,81D517-D3O D3O,D3O,5,1,...,BOYS WEAR,AW22,WLL,JEANS,2995,12.75,-381.8625,2613.1375,RSTIDL/2211/1026,2995


In [ ]:
import pandas as pd
import os

def process_excel_folder(folder_path, rename_dictionary, preview_rows=100):
    """
    Replicates the Power Query M logic in Pandas.
    
    Parameters:
        folder_path (str): Path to folder containing Excel files
        rename_dictionary (dict): Column rename mapping
        preview_rows (int): Number of rows to scan for header detection
        
    Returns:
        pd.DataFrame: Combined cleaned dataframe
    """
    all_data = []

    # Loop through all Excel files in folder
    for file in os.listdir(folder_path):
        if file.endswith((".xlsx", ".xls")):
            
            file_path = os.path.join(folder_path, file)
            excel_file = pd.ExcelFile(file_path)
        
            # Loop through all sheets
            for sheet_name in excel_file.sheet_names:
                df_raw = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
                
                # ----------- HEADER DETECTION LOGIC -----------
                preview = df_raw.head(preview_rows)
                # Count legitimate columns (non-null in column)
                legit_col_count = preview.apply(
                    lambda col: col.dropna().shape[0] > 0
                ).sum()

                # Count distinct non-null values row-wise
                valid_header_counts = preview.apply(
                    lambda row: len(set(row.dropna())), axis=1
                )
                # Find row where header likely exists
                try:
                    row_to_skip = valid_header_counts.tolist().index(legit_col_count)
                except ValueError:
                    continue  # Skip sheet if header not detected

                # Skip rows and promote header
                df = df_raw.iloc[row_to_skip:].reset_index(drop=True)
                df.columns = df.iloc[0]
                df = df[1:].reset_index(drop=True)

                # Remove completely empty columns
                df = df.dropna(axis=1, how='all')

                # ----------- COLUMN RENAME -----------
                df = df.rename(columns=rename_dictionary)

                # Optional: Keep only standardized columns
                keep_cols = ["Date", "Invoice", "Amount"]
                df = df[[col for col in df.columns if col in keep_cols]]

                # Add source file info (optional but useful)
                df["Source_File"] = file
                df["Sheet_Name"] = sheet_name

                all_data.append(df)

    # Combine all files
    if all_data:
        final_df = pd.concat(all_data, ignore_index=True)
        return final_df.dropna()
    else:
        return pd.DataFrame()


In [41]:
rename_dictionary = {
    "BILL Date": "Date",
    "BILL NO.": "Invoice",
    "NET AMOUNT": "Amount",
    "SLS Bill Date": "Date",
    "SLS Bill No": "Invoice",
    "NET SLS REALIZED VALUE": "Amount",
    "BILL_DATE": "Date",
    "BILL_NO": "Invoice",
    "NET_AMT": "Amount",
    "Voucher Date": "Date",
    "Voucher No": "Invoice",
    "Total Value (MRP)": "Amount",
    "BILL DATE": "Date",
    "NET_AMOUNT": "Amount"
}
rename_dictionary

{'BILL Date': 'Date',
 'BILL NO.': 'Invoice',
 'NET AMOUNT': 'Amount',
 'SLS Bill Date': 'Date',
 'SLS Bill No': 'Invoice',
 'NET SLS REALIZED VALUE': 'Amount',
 'BILL_DATE': 'Date',
 'BILL_NO': 'Invoice',
 'NET_AMT': 'Amount',
 'Voucher Date': 'Date',
 'Voucher No': 'Invoice',
 'Total Value (MRP)': 'Amount',
 'BILL DATE': 'Date',
 'NET_AMOUNT': 'Amount'}

In [ ]:
  all_data = []

    # Loop through all Excel files in folder
    for file in os.listdir(folder_path):
        if file.endswith((".xlsx", ".xls")):
            
            file_path = os.path.join(folder_path, file)
            excel_file = pd.ExcelFile(file_path)
        
            # Loop through all sheets
            for sheet_name in excel_file.sheet_names:
                df_raw = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
                
                # ----------- HEADER DETECTION LOGIC -----------
                preview = df_raw.head(preview_rows)
                # Count legitimate columns (non-null in column)
                legit_col_count = preview.apply(
                    lambda col: col.dropna().shape[0] > 0
                ).sum()

                # Count distinct non-null values row-wise
                valid_header_counts = preview.apply(
                    lambda row: len(set(row.dropna())), axis=1
                )
                # Find row where header likely exists
                try:
                    row_to_skip = valid_header_counts.tolist().index(legit_col_count)
                except ValueError:
                    continue  # Skip sheet if header not detected

                # Skip rows and promote header
                df = df_raw.iloc[row_to_skip:].reset_index(drop=True)
                df.columns = df.iloc[0]
                df = df[1:].reset_index(drop=True)

                # Remove completely empty columns
                df = df.dropna(axis=1, how='all')

                # ----------- COLUMN RENAME -----------
                df = df.rename(columns=rename_dictionary)

                # Optional: Keep only standardized columns
                keep_cols = ["Date", "Invoice", "Amount"]
                df = df[[col for col in df.columns if col in keep_cols]]

                # Add source file info (optional but useful)
                df["Source_File"] = file
                df["Sheet_Name"] = sheet_name

                all_data.append(df)

In [8]:
data

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22
0,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...
1,SNO.,BILL Date,BILL NO.,AGENT NAME,COMPANY NAME,ITEM CODE,ITEM NAME,SHADE NAME,PACK / SIZE,SALE QUANTITY,...,CATEGORY.,SEASON.,CODE.,SUB CATEGORY.,GROSS AMOUNT,CD(%),CD VALUE,NET AMOUNT,LOT NUMBER,M.R.P.
2,1,01/12/2022,ASV-24598,NIL,LEVI`S,633731222368,8XR511-508 508,508,7X,1,...,BOYS WEAR,AW22,WLL,JEANS,2095,13.75,-288.0625,1806.9375,RSTIDL/2211/1026,2095
3,2,01/12/2022,ASV-24598,NIL,NIKE,825663750167,86J843-U90 `U90,`U90,7,1,...,BOYS WEAR,AW22,WLL,SWEATSHIRT,2995,13.75,-411.8125,2583.1875,RSTIDL/2209/0603,2995
4,3,01/12/2022,ASV-24600,NIL,NIKE,742728718507,76H992-U1A `U1A,`U1A,2T,-1,...,BOYS WEAR,AW22,WLL,JOGGERS,-1695,NaN,0,-1695,RSTIDL/2209/0603,1695
5,4,01/12/2022,ASV-24600,NIL,NIKE,742728721637,76H926 U9J,U9J,3T,1,...,BOYS WEAR,AW21,WLI,JOGGER,1695,NaN,0,1695,RSTIDL/2209/0603,1695
6,5,01/12/2022,ASV-24604,NIL,JORDAN,807421814823,95B239-023 23,23,L,1,...,BOYS WEAR,AW22,WLL,T-SHIRT,1695,12.2,-206.79,1488.21,RSTIDL/2207/2598,1695
7,6,01/12/2022,ASV-24604,NIL,JORDAN,825664083455,95B958-023 23,23,L,1,...,BOYS WEAR,AW22,WLL,JACKET,6495,12.2,-792.39,5702.61,RSTIDL/2211/1026,6495
8,7,02/12/2022,ASV-24778,NIL,NIKE,677838669718,36G461-W6U W6U,W6U,4,1,...,GIRLS WEAR,AW22,WLL,JACKET,5995,11.67,-699.6165,5295.3835,RSTIDL/2211/1026,5995
9,8,03/12/2022,ASV-24853,NIL,LEVI`S,825663570468,81D517-D3O D3O,D3O,5,1,...,BOYS WEAR,AW22,WLL,JEANS,2995,12.75,-381.8625,2613.1375,RSTIDL/2211/1026,2995


In [9]:
data.head(100)

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22
0,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...
1,SNO.,BILL Date,BILL NO.,AGENT NAME,COMPANY NAME,ITEM CODE,ITEM NAME,SHADE NAME,PACK / SIZE,SALE QUANTITY,...,CATEGORY.,SEASON.,CODE.,SUB CATEGORY.,GROSS AMOUNT,CD(%),CD VALUE,NET AMOUNT,LOT NUMBER,M.R.P.
2,1,01/12/2022,ASV-24598,NIL,LEVI`S,633731222368,8XR511-508 508,508,7X,1,...,BOYS WEAR,AW22,WLL,JEANS,2095,13.75,-288.0625,1806.9375,RSTIDL/2211/1026,2095
3,2,01/12/2022,ASV-24598,NIL,NIKE,825663750167,86J843-U90 `U90,`U90,7,1,...,BOYS WEAR,AW22,WLL,SWEATSHIRT,2995,13.75,-411.8125,2583.1875,RSTIDL/2209/0603,2995
4,3,01/12/2022,ASV-24600,NIL,NIKE,742728718507,76H992-U1A `U1A,`U1A,2T,-1,...,BOYS WEAR,AW22,WLL,JOGGERS,-1695,NaN,0,-1695,RSTIDL/2209/0603,1695
5,4,01/12/2022,ASV-24600,NIL,NIKE,742728721637,76H926 U9J,U9J,3T,1,...,BOYS WEAR,AW21,WLI,JOGGER,1695,NaN,0,1695,RSTIDL/2209/0603,1695
6,5,01/12/2022,ASV-24604,NIL,JORDAN,807421814823,95B239-023 23,23,L,1,...,BOYS WEAR,AW22,WLL,T-SHIRT,1695,12.2,-206.79,1488.21,RSTIDL/2207/2598,1695
7,6,01/12/2022,ASV-24604,NIL,JORDAN,825664083455,95B958-023 23,23,L,1,...,BOYS WEAR,AW22,WLL,JACKET,6495,12.2,-792.39,5702.61,RSTIDL/2211/1026,6495
8,7,02/12/2022,ASV-24778,NIL,NIKE,677838669718,36G461-W6U W6U,W6U,4,1,...,GIRLS WEAR,AW22,WLL,JACKET,5995,11.67,-699.6165,5295.3835,RSTIDL/2211/1026,5995
9,8,03/12/2022,ASV-24853,NIL,LEVI`S,825663570468,81D517-D3O D3O,D3O,5,1,...,BOYS WEAR,AW22,WLL,JEANS,2995,12.75,-381.8625,2613.1375,RSTIDL/2211/1026,2995


In [ ]:
data.apply( lambda col: col.dropna().shape[0] > 0

In [12]:
data['Unnamed: 0'].dropna().shape[0]>0

11

In [16]:
legit_col_count = data.apply(
                    lambda col: col.dropna().shape[0] > 0
                ).sum()
legit_col_count

np.int64(23)

In [15]:
data.apply(
                    lambda col: col.dropna().shape[0] > 0
                ).sum()

np.int64(23)

## Valiod Headers Counts 

In [26]:
valid_header_counts = data.apply(
                    lambda row: len(set(row.dropna())), axis=1
                )

In [17]:
data.apply(lambda row : len(set(row.dropna()))

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22
0,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...,BARND + CATEGORY WISE SALE REPORT NEW From 01/...
1,SNO.,BILL Date,BILL NO.,AGENT NAME,COMPANY NAME,ITEM CODE,ITEM NAME,SHADE NAME,PACK / SIZE,SALE QUANTITY,...,CATEGORY.,SEASON.,CODE.,SUB CATEGORY.,GROSS AMOUNT,CD(%),CD VALUE,NET AMOUNT,LOT NUMBER,M.R.P.
2,1,01/12/2022,ASV-24598,NIL,LEVI`S,633731222368,8XR511-508 508,508,7X,1,...,BOYS WEAR,AW22,WLL,JEANS,2095,13.75,-288.0625,1806.9375,RSTIDL/2211/1026,2095
3,2,01/12/2022,ASV-24598,NIL,NIKE,825663750167,86J843-U90 `U90,`U90,7,1,...,BOYS WEAR,AW22,WLL,SWEATSHIRT,2995,13.75,-411.8125,2583.1875,RSTIDL/2209/0603,2995
4,3,01/12/2022,ASV-24600,NIL,NIKE,742728718507,76H992-U1A `U1A,`U1A,2T,-1,...,BOYS WEAR,AW22,WLL,JOGGERS,-1695,NaN,0,-1695,RSTIDL/2209/0603,1695
5,4,01/12/2022,ASV-24600,NIL,NIKE,742728721637,76H926 U9J,U9J,3T,1,...,BOYS WEAR,AW21,WLI,JOGGER,1695,NaN,0,1695,RSTIDL/2209/0603,1695
6,5,01/12/2022,ASV-24604,NIL,JORDAN,807421814823,95B239-023 23,23,L,1,...,BOYS WEAR,AW22,WLL,T-SHIRT,1695,12.2,-206.79,1488.21,RSTIDL/2207/2598,1695
7,6,01/12/2022,ASV-24604,NIL,JORDAN,825664083455,95B958-023 23,23,L,1,...,BOYS WEAR,AW22,WLL,JACKET,6495,12.2,-792.39,5702.61,RSTIDL/2211/1026,6495
8,7,02/12/2022,ASV-24778,NIL,NIKE,677838669718,36G461-W6U W6U,W6U,4,1,...,GIRLS WEAR,AW22,WLL,JACKET,5995,11.67,-699.6165,5295.3835,RSTIDL/2211/1026,5995
9,8,03/12/2022,ASV-24853,NIL,LEVI`S,825663570468,81D517-D3O D3O,D3O,5,1,...,BOYS WEAR,AW22,WLL,JEANS,2995,12.75,-381.8625,2613.1375,RSTIDL/2211/1026,2995


In [22]:
len(set(data.iloc[0].values))

2

In [25]:
len(set(data.iloc[1].values))

23

In [27]:
valid_header_counts

0      2
1     23
2     21
3     21
4     20
5     19
6     21
7     21
8     21
9     21
10    21
11     5
dtype: int64

In [28]:
valid_header_counts.tolist()

[2, 23, 21, 21, 20, 19, 21, 21, 21, 21, 21, 5]

In [29]:
legit_col_count

np.int64(23)

In [30]:
valid_header_counts.tolist().index(legit_col_count)
              

1

In [33]:
data=data.iloc[valid_header_counts.tolist().index(legit_col_count):]

In [36]:
data.columns=data.iloc[0]

In [37]:
data

1,SNO.,BILL Date,BILL NO.,AGENT NAME,COMPANY NAME,ITEM CODE,ITEM NAME,SHADE NAME,PACK / SIZE,SALE QUANTITY,...,CATEGORY.,SEASON.,CODE.,SUB CATEGORY.,GROSS AMOUNT,CD(%),CD VALUE,NET AMOUNT,LOT NUMBER,M.R.P.
1,SNO.,BILL Date,BILL NO.,AGENT NAME,COMPANY NAME,ITEM CODE,ITEM NAME,SHADE NAME,PACK / SIZE,SALE QUANTITY,...,CATEGORY.,SEASON.,CODE.,SUB CATEGORY.,GROSS AMOUNT,CD(%),CD VALUE,NET AMOUNT,LOT NUMBER,M.R.P.
2,1,01/12/2022,ASV-24598,NIL,LEVI`S,633731222368,8XR511-508 508,508,7X,1,...,BOYS WEAR,AW22,WLL,JEANS,2095,13.75,-288.0625,1806.9375,RSTIDL/2211/1026,2095
3,2,01/12/2022,ASV-24598,NIL,NIKE,825663750167,86J843-U90 `U90,`U90,7,1,...,BOYS WEAR,AW22,WLL,SWEATSHIRT,2995,13.75,-411.8125,2583.1875,RSTIDL/2209/0603,2995
4,3,01/12/2022,ASV-24600,NIL,NIKE,742728718507,76H992-U1A `U1A,`U1A,2T,-1,...,BOYS WEAR,AW22,WLL,JOGGERS,-1695,NaN,0,-1695,RSTIDL/2209/0603,1695
5,4,01/12/2022,ASV-24600,NIL,NIKE,742728721637,76H926 U9J,U9J,3T,1,...,BOYS WEAR,AW21,WLI,JOGGER,1695,NaN,0,1695,RSTIDL/2209/0603,1695
6,5,01/12/2022,ASV-24604,NIL,JORDAN,807421814823,95B239-023 23,23,L,1,...,BOYS WEAR,AW22,WLL,T-SHIRT,1695,12.2,-206.79,1488.21,RSTIDL/2207/2598,1695
7,6,01/12/2022,ASV-24604,NIL,JORDAN,825664083455,95B958-023 23,23,L,1,...,BOYS WEAR,AW22,WLL,JACKET,6495,12.2,-792.39,5702.61,RSTIDL/2211/1026,6495
8,7,02/12/2022,ASV-24778,NIL,NIKE,677838669718,36G461-W6U W6U,W6U,4,1,...,GIRLS WEAR,AW22,WLL,JACKET,5995,11.67,-699.6165,5295.3835,RSTIDL/2211/1026,5995
9,8,03/12/2022,ASV-24853,NIL,LEVI`S,825663570468,81D517-D3O D3O,D3O,5,1,...,BOYS WEAR,AW22,WLL,JEANS,2995,12.75,-381.8625,2613.1375,RSTIDL/2211/1026,2995
10,9,03/12/2022,ASV-24853,NIL,JORDAN,742728817613,85A715-023 `023,`023,6,1,...,BOYS WEAR,AW22,WLL,SWEATSHIRT,2495,12.75,-318.1125,2176.8875,RSTIDL/2209/0603,2495


In [40]:
data[1:].reset_index(drop=True)


1,SNO.,BILL Date,BILL NO.,AGENT NAME,COMPANY NAME,ITEM CODE,ITEM NAME,SHADE NAME,PACK / SIZE,SALE QUANTITY,...,CATEGORY.,SEASON.,CODE.,SUB CATEGORY.,GROSS AMOUNT,CD(%),CD VALUE,NET AMOUNT,LOT NUMBER,M.R.P.
0,1,01/12/2022,ASV-24598,NIL,LEVI`S,633731222368,8XR511-508 508,508,7X,1,...,BOYS WEAR,AW22,WLL,JEANS,2095,13.75,-288.0625,1806.9375,RSTIDL/2211/1026,2095
1,2,01/12/2022,ASV-24598,NIL,NIKE,825663750167,86J843-U90 `U90,`U90,7,1,...,BOYS WEAR,AW22,WLL,SWEATSHIRT,2995,13.75,-411.8125,2583.1875,RSTIDL/2209/0603,2995
2,3,01/12/2022,ASV-24600,NIL,NIKE,742728718507,76H992-U1A `U1A,`U1A,2T,-1,...,BOYS WEAR,AW22,WLL,JOGGERS,-1695,NaN,0,-1695,RSTIDL/2209/0603,1695
3,4,01/12/2022,ASV-24600,NIL,NIKE,742728721637,76H926 U9J,U9J,3T,1,...,BOYS WEAR,AW21,WLI,JOGGER,1695,NaN,0,1695,RSTIDL/2209/0603,1695
4,5,01/12/2022,ASV-24604,NIL,JORDAN,807421814823,95B239-023 23,23,L,1,...,BOYS WEAR,AW22,WLL,T-SHIRT,1695,12.2,-206.79,1488.21,RSTIDL/2207/2598,1695
5,6,01/12/2022,ASV-24604,NIL,JORDAN,825664083455,95B958-023 23,23,L,1,...,BOYS WEAR,AW22,WLL,JACKET,6495,12.2,-792.39,5702.61,RSTIDL/2211/1026,6495
6,7,02/12/2022,ASV-24778,NIL,NIKE,677838669718,36G461-W6U W6U,W6U,4,1,...,GIRLS WEAR,AW22,WLL,JACKET,5995,11.67,-699.6165,5295.3835,RSTIDL/2211/1026,5995
7,8,03/12/2022,ASV-24853,NIL,LEVI`S,825663570468,81D517-D3O D3O,D3O,5,1,...,BOYS WEAR,AW22,WLL,JEANS,2995,12.75,-381.8625,2613.1375,RSTIDL/2211/1026,2995
8,9,03/12/2022,ASV-24853,NIL,JORDAN,742728817613,85A715-023 `023,`023,6,1,...,BOYS WEAR,AW22,WLL,SWEATSHIRT,2495,12.75,-318.1125,2176.8875,RSTIDL/2209/0603,2495
9,NaN,NaN,GRAND TOTALS,NaN,NaN,NaN,NaN,NaN,NaN,182,...,NaN,NaN,NaN,NaN,24765,NaN,-3098.6465,21666.3535,NaN,NaN


In [47]:
data = data.rename(columns=rename_dictionary)
data

1,SNO.,Date,Invoice,AGENT NAME,COMPANY NAME,ITEM CODE,ITEM NAME,SHADE NAME,PACK / SIZE,SALE QUANTITY,...,CATEGORY.,SEASON.,CODE.,SUB CATEGORY.,GROSS AMOUNT,CD(%),CD VALUE,Amount,LOT NUMBER,M.R.P.
1,SNO.,BILL Date,BILL NO.,AGENT NAME,COMPANY NAME,ITEM CODE,ITEM NAME,SHADE NAME,PACK / SIZE,SALE QUANTITY,...,CATEGORY.,SEASON.,CODE.,SUB CATEGORY.,GROSS AMOUNT,CD(%),CD VALUE,NET AMOUNT,LOT NUMBER,M.R.P.
2,1,01/12/2022,ASV-24598,NIL,LEVI`S,633731222368,8XR511-508 508,508,7X,1,...,BOYS WEAR,AW22,WLL,JEANS,2095,13.75,-288.0625,1806.9375,RSTIDL/2211/1026,2095
3,2,01/12/2022,ASV-24598,NIL,NIKE,825663750167,86J843-U90 `U90,`U90,7,1,...,BOYS WEAR,AW22,WLL,SWEATSHIRT,2995,13.75,-411.8125,2583.1875,RSTIDL/2209/0603,2995
4,3,01/12/2022,ASV-24600,NIL,NIKE,742728718507,76H992-U1A `U1A,`U1A,2T,-1,...,BOYS WEAR,AW22,WLL,JOGGERS,-1695,NaN,0,-1695,RSTIDL/2209/0603,1695
5,4,01/12/2022,ASV-24600,NIL,NIKE,742728721637,76H926 U9J,U9J,3T,1,...,BOYS WEAR,AW21,WLI,JOGGER,1695,NaN,0,1695,RSTIDL/2209/0603,1695
6,5,01/12/2022,ASV-24604,NIL,JORDAN,807421814823,95B239-023 23,23,L,1,...,BOYS WEAR,AW22,WLL,T-SHIRT,1695,12.2,-206.79,1488.21,RSTIDL/2207/2598,1695
7,6,01/12/2022,ASV-24604,NIL,JORDAN,825664083455,95B958-023 23,23,L,1,...,BOYS WEAR,AW22,WLL,JACKET,6495,12.2,-792.39,5702.61,RSTIDL/2211/1026,6495
8,7,02/12/2022,ASV-24778,NIL,NIKE,677838669718,36G461-W6U W6U,W6U,4,1,...,GIRLS WEAR,AW22,WLL,JACKET,5995,11.67,-699.6165,5295.3835,RSTIDL/2211/1026,5995
9,8,03/12/2022,ASV-24853,NIL,LEVI`S,825663570468,81D517-D3O D3O,D3O,5,1,...,BOYS WEAR,AW22,WLL,JEANS,2995,12.75,-381.8625,2613.1375,RSTIDL/2211/1026,2995
10,9,03/12/2022,ASV-24853,NIL,JORDAN,742728817613,85A715-023 `023,`023,6,1,...,BOYS WEAR,AW22,WLL,SWEATSHIRT,2495,12.75,-318.1125,2176.8875,RSTIDL/2209/0603,2495


In [44]:
 keep_cols = ["Date", "Invoice", "Amount"]

In [48]:
data[keep_cols]

1,Date,Invoice,Amount
1,BILL Date,BILL NO.,NET AMOUNT
2,01/12/2022,ASV-24598,1806.9375
3,01/12/2022,ASV-24598,2583.1875
4,01/12/2022,ASV-24600,-1695
5,01/12/2022,ASV-24600,1695
6,01/12/2022,ASV-24604,1488.21
7,01/12/2022,ASV-24604,5702.61
8,02/12/2022,ASV-24778,5295.3835
9,03/12/2022,ASV-24853,2613.1375
10,03/12/2022,ASV-24853,2176.8875


In [ ]:
 # Skip rows and promote header
                df = df_raw.iloc[row_to_skip:].reset_index(drop=True)
                df.columns = df.iloc[0]
                df = df[1:].reset_index(drop=True)

                # Remove completely empty columns
                df = df.dropna(axis=1, how='all')

                # ----------- COLUMN RENAME -----------
                df = df.rename(columns=rename_dictionary)

                # Optional: Keep only standardized columns
                keep_cols = ["Date", "Invoice", "Amount"]
                df = df[[col for col in df.columns if col in keep_cols]]

                # Add source file info (optional but useful)
                df["Source_File"] = file
                df["Sheet_Name"] = sheet_name

# Finakl 

In [49]:
import pandas as pd
import os

def process_excel_folder(folder_path, rename_dictionary, preview_rows=100):
    all_data = []
    for file in os.listdir(folder_path):
        if file.endswith((".xlsx", ".xls")):
            
            file_path = os.path.join(folder_path, file)
            excel_file = pd.ExcelFile(file_path)
        
            # Loop through all sheets
            for sheet_name in excel_file.sheet_names:
                df_raw = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
                
                #  HEADER DETECTION LOGIC 
                preview = df_raw.head(preview_rows)
                legit_col_count = preview.apply(
                    lambda col: col.dropna().shape[0] > 0
                ).sum()
                valid_header_counts = preview.apply(
                    lambda row: len(set(row.dropna())), axis=1
                )
                try:
                    row_to_skip = valid_header_counts.tolist().index(legit_col_count)
                except ValueError:
                    continue  # Skip sheet if header not detected
                df = df_raw.iloc[row_to_skip:].reset_index(drop=True)
                df.columns = df.iloc[0]
                df = df[1:].reset_index(drop=True)
                df = df.dropna(axis=1, how='all')

                #  COLUMN RENAME 
                df = df.rename(columns=rename_dictionary)
                keep_cols = ["Date", "Invoice", "Amount"]
                df = df[[col for col in df.columns if col in keep_cols]]
                df["Source_File"] = file
                df["Sheet_Name"] = sheet_name

                all_data.append(df)

    # Combine all files
    if all_data:
        final_df = pd.concat(all_data, ignore_index=True)
        return final_df.dropna()
    else:
        return pd.DataFrame()


In [50]:
rename_dictionary = {
    "BILL Date": "Date",
    "BILL NO.": "Invoice",
    "NET AMOUNT": "Amount",
    "SLS Bill Date": "Date",
    "SLS Bill No": "Invoice",
    "NET SLS REALIZED VALUE": "Amount",
    "BILL_DATE": "Date",
    "BILL_NO": "Invoice",
    "NET_AMT": "Amount",
    "Voucher Date": "Date",
    "Voucher No": "Invoice",
    "Total Value (MRP)": "Amount",
    "BILL DATE": "Date",
    "NET_AMOUNT": "Amount"
}
rename_dictionary

{'BILL Date': 'Date',
 'BILL NO.': 'Invoice',
 'NET AMOUNT': 'Amount',
 'SLS Bill Date': 'Date',
 'SLS Bill No': 'Invoice',
 'NET SLS REALIZED VALUE': 'Amount',
 'BILL_DATE': 'Date',
 'BILL_NO': 'Invoice',
 'NET_AMT': 'Amount',
 'Voucher Date': 'Date',
 'Voucher No': 'Invoice',
 'Total Value (MRP)': 'Amount',
 'BILL DATE': 'Date',
 'NET_AMOUNT': 'Amount'}

In [52]:
 process_excel_folder(r"R:\adarsha\power_bi_legit_columns\Files-solution (1)\Files",rename_dictionary)

,Date,Invoice,Amount,Source_File,Sheet_Name
0,01/12/2022,ASV-24598,1806.9375,ABC Store.xlsx,Report
1,01/12/2022,ASV-24598,2583.1875,ABC Store.xlsx,Report
2,01/12/2022,ASV-24600,-1695,ABC Store.xlsx,Report
3,01/12/2022,ASV-24600,1695,ABC Store.xlsx,Report
4,01/12/2022,ASV-24604,1488.21,ABC Store.xlsx,Report
5,01/12/2022,ASV-24604,5702.61,ABC Store.xlsx,Report
6,02/12/2022,ASV-24778,5295.3835,ABC Store.xlsx,Report
7,03/12/2022,ASV-24853,2613.1375,ABC Store.xlsx,Report
8,03/12/2022,ASV-24853,2176.8875,ABC Store.xlsx,Report
10,26-12-2022,MBGF-0034297,1497,Bobby Shop.xlsx,Sheet1


In [53]:
import os
os.getcwd()

'R:\\adarsha\\power_bi_legit_columns\\Files-solution (1)\\Files'